### Import Package

In [1]:
import os
import pandas as pd
import numpy as np
import tensorflow as tf
from datasets import load_dataset
from tensorflow.keras.callbacks import ModelCheckpoint
from transformers import AutoTokenizer
from transformers import create_optimizer
from transformers import TFAutoModelForSequenceClassification
from transformers import DataCollatorWithPadding

2022-05-31 19:21:31.227529: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcudart.so.11.0'; dlerror: libcudart.so.11.0: cannot open shared object file: No such file or directory
2022-05-31 19:21:31.227547: I tensorflow/stream_executor/cuda/cudart_stub.cc:29] Ignore above cudart dlerror if you do not have a GPU set up on your machine.
/usr/lib/python3/dist-packages/requests/__init__.py:89: RequestsDependencyWarning: urllib3 (1.26.9) or chardet (3.0.4) doesn't match a supported version!
  warnings.warn("urllib3 ({}) or chardet ({}) doesn't match a supported "
/home/feng/.local/lib/python3.8/site-packages/tqdm/auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Read and Split Dataset

In [2]:
training_file = "dataset/train.csv"
test_file = "dataset/test.csv"
output_dir = './model2_outputs'
batch_size = 16

In [3]:
df = load_dataset('csv', data_files = [training_file])
df = df['train'].train_test_split(test_size = 0.1)
df['valid'] = df['test']
df['test'] = load_dataset('csv', data_files = [test_file])['train']

Using custom data configuration default-dacd6034bd90bc38
Reusing dataset csv (/home/feng/.cache/huggingface/datasets/csv/default-dacd6034bd90bc38/0.0.0/433e0ccc46f9880962cc2b12065189766fbb2bee57a221866138fb9203c83519)
100%|██████████| 1/1 [00:00<00:00, 991.80it/s]
Using custom data configuration default-60f927a822212e41
Reusing dataset csv (/home/feng/.cache/huggingface/datasets/csv/default-60f927a822212e41/0.0.0/433e0ccc46f9880962cc2b12065189766fbb2bee57a221866138fb9203c83519)
100%|██████████| 1/1 [00:00<00:00, 691.56it/s]


In [4]:
pd.DataFrame(df['train'])

,id,keyword,location,text,target
0,10768,wreckage,None,Wreckage is MH370: Najib\nhttp://t.co/iidKC0jS...,1
1,7878,quarantined,None,Alabama firefighters quarantined after possibl...,1
2,4061,displaced,Seattle,The year is 2065 and the national society of m...,0
3,3237,deluged,None,Businesses are deluged with invoices. Make you...,0
4,4373,earthquake,"Hawaii, USA",USGS reports a M1.94 #earthquake 5km S of Volc...,1
...,...,...,...,...,...
6846,1990,bush%20fires,None,28 Oct 1895: 'Bush Fires.' http://t.co/zCKXtFc9PT,1
6847,5137,fatal,None,11-Year-Old Boy Charged With Manslaughter of T...,1
6848,1618,bombed,None,Me trying to pass lax with my family ends up b...,0
6849,3130,debris,None,Debris found on Reunion Island comes from MH37...,1


In [5]:
pd.DataFrame(df['valid'])

,id,keyword,location,text,target
0,3940,devastated,"Chicago, IL",@Keegan172 I'm devastated,0
1,7642,pandemonium,Everywhere,Pandemonium In Aba As Woman Delivers Baby With...,1
2,2882,damage,"Pontevedra, Galicia",#NP Metallica - Damage Inc,0
3,3428,derail,London,Don't let the #tubestrike derail your mood and...,0
4,797,battle,None,CIVIL WAR GENERAL BATTLE BULL RUN HERO COLONEL...,1
...,...,...,...,...,...
757,517,army,New York,INFANTRY Mens Lume Dial Army Analog Quartz Wri...,0
758,10072,typhoon,None,Typhoon Soudelor: When will it hit Taiwan ÛÒ ...,1
759,3705,destroyed,None,Hero you can 't swim lonely guy help me my sol...,0
760,983,blazing,"Intramuros, Manila",Come and join us Tomorrow!\nAugust 7 2015 at T...,0


In [6]:
pd.DataFrame(df['test'])

,id,keyword,location,text
0,0,None,None,Just happened a terrible car crash
1,2,None,None,"Heard about #earthquake is different cities, s..."
2,3,None,None,"there is a forest fire at spot pond, geese are..."
3,9,None,None,Apocalypse lighting. #Spokane #wildfires
4,11,None,None,Typhoon Soudelor kills 28 in China and Taiwan
...,...,...,...,...
3258,10861,None,None,EARTHQUAKE SAFETY LOS ANGELES ÛÒ SAFETY FASTE...
3259,10865,None,None,Storm in RI worse than last hurricane. My city...
3260,10868,None,None,Green Line derailment in Chicago http://t.co/U...
3261,10874,None,None,MEG issues Hazardous Weather Outlook (HWO) htt...


In [7]:
os.environ["TOKENIZERS_PARALLELISM"] = "false" # Stop Warnings
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

def tokenize_function(examples):
    return tokenizer(examples["text"], padding = "max_length", truncation = True)

pre_tokenizer_columns = set(df["train"].features)
encoded_df = df.map(tokenize_function, batched = True)
tokenizer_columns = list(set(encoded_df["train"].features) - pre_tokenizer_columns)
print("Columns added by tokenizer:", tokenizer_columns)

100%|██████████| 1/1 [00:00<00:00,  9.69ba/s]

Columns added by tokenizer: ['input_ids', 'attention_mask']


In [8]:
df

DatasetDict({
    train: Dataset({
        features: ['id', 'keyword', 'location', 'text', 'target'],
        num_rows: 6851
    })
    test: Dataset({
        features: ['id', 'keyword', 'location', 'text'],
        num_rows: 3263
    })
    valid: Dataset({
        features: ['id', 'keyword', 'location', 'text', 'target'],
        num_rows: 762
    })
})

In [9]:
encoded_df

DatasetDict({
    train: Dataset({
        features: ['id', 'keyword', 'location', 'text', 'target', 'input_ids', 'attention_mask'],
        num_rows: 6851
    })
    test: Dataset({
        features: ['id', 'keyword', 'location', 'text', 'input_ids', 'attention_mask'],
        num_rows: 3263
    })
    valid: Dataset({
        features: ['id', 'keyword', 'location', 'text', 'target', 'input_ids', 'attention_mask'],
        num_rows: 762
    })
})

In [10]:
data_collator = DataCollatorWithPadding(tokenizer = tokenizer, return_tensors = "tf")

train_df = encoded_df['train'].to_tf_dataset(
    columns = tokenizer_columns,
    label_cols = ["target"],
    shuffle = True,
    collate_fn = data_collator,
    batch_size = batch_size,
)

val_df = encoded_df['valid'].to_tf_dataset(
    columns = tokenizer_columns,
    label_cols = ["target"],
    shuffle = False,
    batch_size = batch_size,
    collate_fn = data_collator,
)

test_df = encoded_df['test'].to_tf_dataset(
    columns = tokenizer_columns,
    shuffle = False,
    batch_size = batch_size,
    collate_fn = data_collator,
)

test_df

2022-05-31 19:21:44.740575: I tensorflow/stream_executor/cuda/cuda_gpu_executor.cc:936] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero
2022-05-31 19:21:44.741135: I tensorflow/stream_executor/cuda/cuda_gpu_executor.cc:936] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero
2022-05-31 19:21:44.741691: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcudart.so.11.0'; dlerror: libcudart.so.11.0: cannot open shared object file: No such file or directory
2022-05-31 19:21:44.741745: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcublas.so.11'; dlerror: libcublas.so.11: cannot open shared object file: No such file or directory
2022-05-31 19:21:44.741804: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not lo

<PrefetchDataset element_spec={'input_ids': TensorSpec(shape=(None, None), dtype=tf.int64, name=None), 'attention_mask': TensorSpec(shape=(None, None), dtype=tf.int64, name=None)}>

In [11]:
model = TFAutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased", num_labels = 2
)

model.summary()

2022-05-31 19:21:46.657868: W tensorflow/python/util/util.cc:368] Sets are not currently considered sequences, but this may change in the future, so consider avoiding using them.
Some layers from the model checkpoint at distilbert-base-uncased were not used when initializing TFDistilBertForSequenceClassification: ['activation_13', 'vocab_transform', 'vocab_layer_norm', 'vocab_projector']
- This IS expected if you are initializing TFDistilBertForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing TFDistilBertForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Some layers of TFDistilBertForSequenceClassification were not initialized from the model checkpoint 

In [12]:
num_epochs = 3
batches_per_epoch = len(encoded_df["train"]) // batch_size
total_train_steps = int(batches_per_epoch * num_epochs)

optimizer, schedule = create_optimizer(
    init_lr = 2e-5, num_warmup_steps = 0, num_train_steps = total_train_steps
)
loss = tf.keras.losses.SparseCategoricalCrossentropy(from_logits = True)
model.compile(optimizer = optimizer, loss = loss, metrics = ['accuracy'])

In [13]:
if not os.path.exists(output_dir): # If the file directory doesn't already exists,
    os.makedirs(output_dir) # Make it again

checkpoint_callback = ModelCheckpoint(filepath = output_dir + '/weights.{epoch:02d}.hdf5', monitor = 'val_loss', save_best_only = True, save_weights_only = True)

In [14]:
model.fit(
    train_df,
    validation_data = val_df,
    epochs = 3,
    callbacks = [checkpoint_callback],
)

Epoch 1/3
428/428 [==============================] - ETA: 0s - loss: 0.4331 - accuracy: 0.8096

NotImplementedError: Saving the model to HDF5 format requires the model to be a Functional model or a Sequential model. It does not work for subclassed models, because such models are defined via the body of a Python method, which isn't safely serializable. Consider saving to the Tensorflow SavedModel format (by setting save_format="tf") or using `save_weights`.

### Predict and Output Test Dataset

In [ ]:
test_pred = model.predict(test_df)

In [ ]:
submission = pd.read_csv('dataset/sample_submission.csv')
submission['target'] = np.argmax(test_pred.logits, axis = 1)
submission.to_csv('submission.csv', index = False)